<a href="https://colab.research.google.com/github/serendipity-sd/GenAI/blob/main/vizuara_context_engineering/Day2_Context_Engineering_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Day 2: Anatomy of Context — Interactive Lab

**AI Context Engineering Workshop — Session 2**

---

## Learning Objectives

By the end of this lab you will understand how **three layers of context** control LLM output quality:

| Layer | What It Does | Lab |
|-------|-------------|-----|
| **CLAUDE.md / Rules File** | Encodes project conventions so the model follows *your* patterns | Lab 1 |
| **System Prompt** | Constrains behaviour with rules, tone, and edge-case routing | Lab 2 |
| **Few-Shot Examples** | Teaches output format and style by demonstration | Lab 3 |
| **Context Stack** | Combines all three for maximum quality | Lab 4 |
| **Selective Retrieval** | Retrieves *only* the relevant context — saves tokens, keeps quality | Lab 5 |

### How to use this notebook
1. Run the **Setup** cell first (installs the Gemini SDK & sets your API key).
2. Work through each lab **in order** — later labs build on earlier concepts.
3. Each lab has a **🎯 Try It Yourself** cell where you can experiment.

> **LLM Provider:** Google Gemini (`gemini-2.0-flash`) — fast, cheap, and excellent for teaching.

In [ ]:
# ═══ Setup: Install SDK & Configure API Key ═══
!pip install -q google-generativeai

import getpass, textwrap, re
import google.generativeai as genai

API_KEY = getpass.getpass("Enter your Google Gemini API key: ")
genai.configure(api_key=API_KEY)

MODEL_NAME = "gemini-2.0-flash"

def call_gemini(system_prompt: str, user_message: str) -> str:
    """Send a message to Gemini with a system prompt and return the response text."""
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=system_prompt if system_prompt else None,
    )
    response = model.generate_content(user_message)
    return response.text

# Quick sanity check
print(call_gemini("You are a helpful assistant.", "Say 'Hello from Gemini!' and nothing else."))
print(f"\n\u2705 Setup complete — using {MODEL_NAME}")

---
## Lab 1: CLAUDE.md / Rules File

### The Scenario

You’re building a **student grade calculator** in Python. The project has 5 **non-standard conventions**:

| # | Convention | Standard Python | Our Project |
|---|-----------|----------------|-------------|
| 1 | Arithmetic | `float` | `Decimal` from `decimal` module |
| 2 | Letter grades | `if/elif` chain | Lookup table (`dict`) |
| 3 | Return type | plain `dict` | `GradeReport` dataclass |
| 4 | CSV export | f-string / `+` concat | `csv.writer` |
| 5 | Weights | hardcoded `* 0.3` | config dict |

Without a rules file, the LLM writes standard Python that violates **all 5 conventions**.
With a CLAUDE.md, it follows every one.

### Project Structure
```
grade-calculator/
  src/
    calculator.py      # Core grade calculation logic
    models.py          # GradeReport dataclass, StudentRecord
    letter_grades.py   # Grade lookup table (not if/elif!)
    config.py          # Weight configuration (loads from TOML)
    export.py          # CSV export using csv.writer
    cli.py             # Command-line interface
  tests/
    test_calculator.py # Tests for grade calculations
    test_export.py     # Tests for CSV output
    conftest.py        # Shared fixtures (sample students)
  config/
    weights.toml       # Assignment weight configuration
  requirements.txt
```

In [ ]:
# ═══ Lab 1: Data & Scoring ═══

CLAUDE_MD_SOLUTION = {
    "overview": (
        "A student grade calculator that computes weighted final grades, "
        "assigns letter grades, and exports results to CSV. Used by instructors "
        "to process class rosters. Uses precise Decimal arithmetic, dataclass "
        "result objects, and configurable weight schemes."
    ),
    "structure": (
        "- src/calculator.py    \u2014 Core logic: calculate_final_grade() returns GradeReport\n"
        "- src/models.py        \u2014 GradeReport dataclass (frozen=True), StudentRecord\n"
        "- src/letter_grades.py \u2014 GRADE_CUTOFFS lookup table (dict mapping, not if/elif)\n"
        "- src/config.py        \u2014 Loads weights from config/weights.toml via tomllib\n"
        "- src/export.py        \u2014 CSV export using csv.writer (not string concatenation)\n"
        "- tests/               \u2014 Pytest tests with sample student fixtures"
    ),
    "conventions": (
        "- All grade math uses Decimal from the decimal module \u2014 NEVER float\n"
        "- Letter grades determined by GRADE_CUTOFFS lookup table \u2014 NEVER if/elif chains\n"
        "- All results returned as GradeReport dataclass \u2014 NEVER plain dicts\n"
        "- CSV export uses csv.writer or csv.DictWriter \u2014 NEVER string concatenation\n"
        "- Assignment weights loaded from config dict \u2014 NEVER hardcoded numbers"
    ),
    "testing": (
        "- Run: pytest tests/ -v\n"
        "- Use sample_student() fixture from conftest.py for test data\n"
        "- Test edge cases: exact boundaries (89.5 \u2192 B+), zero scores, missing categories\n"
        "- Assert on GradeReport fields, not dict keys\n"
        "- Use Decimal(\"95.5\") not float 95.5 in test values"
    ),
    "patterns": (
        "- Decimal Math: from decimal import Decimal; always Decimal(\"95.5\") not 95.5\n"
        "- Lookup Table: GRADE_CUTOFFS = {Decimal(\"90\"): \"A\", Decimal(\"80\"): \"B\", ...}\n"
        "  Use: letter = next(v for k,v in sorted(table.items(), reverse=True) if score >= k)\n"
        "- GradeReport: @dataclass(frozen=True) with student_name, final_grade, letter_grade fields\n"
        "- Config Weights: weights = {\"homework\": Decimal(\"0.30\"), \"midterm\": Decimal(\"0.30\"), ...}\n"
        "- CSV Export: writer = csv.writer(f); writer.writerow([name, grade, letter])"
    ),
    "mistakes": (
        "- Do NOT use float for grade calculations \u2014 use Decimal for precision\n"
        "- Do NOT use if/elif chains for letter grades \u2014 use the GRADE_CUTOFFS lookup table\n"
        "- Do NOT return plain dicts \u2014 always return GradeReport dataclass instances\n"
        "- Do NOT build CSV with f-strings or + concatenation \u2014 use csv.writer\n"
        "- Do NOT hardcode weights like * 0.3 \u2014 load from config/weights dict"
    ),
}


def build_claude_md(sections: dict[str, str]) -> str:
    """Assemble a CLAUDE.md string from filled-in sections."""
    headings = {
        "overview": "# Project Overview",
        "structure": "# Project Structure",
        "conventions": "# Code Conventions",
        "testing": "# Testing Requirements",
        "patterns": "# Important Patterns",
        "mistakes": "# Common Mistakes to Avoid",
    }
    parts = []
    for key, heading in headings.items():
        if sections.get(key):
            parts.append(f"{heading}\n{sections[key]}")
    return "\n\n".join(parts)


CODING_TASK = (
    "Write a Python function calculate_final_grade(student_name, scores, weights) that:\n"
    "1. Computes a weighted average from the scores dict and weights dict\n"
    "2. Determines the letter grade\n"
    "3. Returns the result\n"
    "\nAlso write a function to export a list of results to CSV, and a pytest test."
)

SCORE_LABELS = [
    ("decimal_math",     "Decimal Math"),
    ("lookup_table",     "Lookup Table"),
    ("dataclass_result", "GradeReport Dataclass"),
    ("csv_writer",       "CSV Writer"),
    ("config_weights",   "Config Weights"),
]


def score_code(response: str) -> dict:
    """Score generated code on 5 project-specific conventions (0 or 5 each, max 25)."""
    lower = response.lower()

    scores = {
        "decimal_math": 5 if any(p in lower for p in ["from decimal", "decimal(", "import decimal"]) else 0,
        "lookup_table": 5 if any(p in lower for p in [
            "grade_cutoff", "grade_table", "grade_map", "letter_grades",
            "grade_scale", "grade_boundar", "thresholds", "grade_range",
            "cutoffs", "grade_thresholds",
        ]) else 0,
        "dataclass_result": 5 if (
            "gradereport" in lower or
            ("dataclass" in lower and "grade" in lower and "class" in lower)
        ) else 0,
        "csv_writer": 5 if any(p in lower for p in ["csv.writer", "csv.dictwriter", "writerow", "writerows"]) else 0,
        "config_weights": 5 if any(p in lower for p in [
            "weights[", "weights.get(", "weights = {", "weights={",
            "weight_config", "category_weights", "assignment_weights",
        ]) else 0,
    }
    scores["total"] = sum(scores.values())
    return scores


print("\u2705 Lab 1 data & scoring loaded")

In [ ]:
# ═══ Lab 1: Run — Without vs With CLAUDE.md ═══

claude_md_text = build_claude_md(CLAUDE_MD_SOLUTION)

# --- Call 1: No context ---
print("\u23f3 Generating code WITHOUT CLAUDE.md context...")
bare_prompt = "You are a helpful Python developer."
response_without = call_gemini(bare_prompt, CODING_TASK)
scores_without = score_code(response_without)

# --- Call 2: With CLAUDE.md context ---
print("\u23f3 Generating code WITH CLAUDE.md context...")
context_prompt = f"""You are a helpful Python developer.

Below is the project's CLAUDE.md rules file. Follow ALL conventions exactly.

---
{claude_md_text}
---"""
response_with = call_gemini(context_prompt, CODING_TASK)
scores_with = score_code(response_with)

# --- Results Table ---
print("\n" + "=" * 60)
print("LAB 1 RESULTS: CLAUDE.md Impact")
print("=" * 60)
print(f"{'Convention':<25} {'Without':>10} {'With':>10}")
print("-" * 45)
for key, label in SCORE_LABELS:
    w = "\u2705" if scores_without[key] == 5 else "\u274c"
    c = "\u2705" if scores_with[key] == 5 else "\u274c"
    print(f"{label:<25} {w:>10} {c:>10}")
print("-" * 45)
print(f"{'TOTAL':<25} {scores_without['total']:>10}/25 {scores_with['total']:>10}/25")
print("\n\u2728 Key Insight: The LLM doesn't know your project's conventions.")
print("   CLAUDE.md is how you TEACH them.")

### Lab 1 Interpretation

**Without** a rules file, the model writes perfectly valid Python — but uses `float`, `if/elif`, plain `dict`, and string concatenation. It has no way to know your project uses `Decimal`, lookup tables, dataclasses, etc.

**With** the CLAUDE.md context, every convention is followed. This is the power of a rules file: it encodes project-specific knowledge that no pre-training can provide.

> **Takeaway:** A well-written CLAUDE.md is the single highest-leverage context engineering technique.

---
## Lab 2: System Prompt Engineering

### The Scenario

Build a customer support chatbot for **CloudStore** (online electronics retailer). Start with a minimal prompt, observe failures, then iteratively add rules across **4 rounds**:

| Round | What We Add | Why |
|-------|------------|-----|
| 1 — Minimal | Identity + purpose only | Baseline — watch it fail |
| 2 — +Rules | Anti-hallucination, tool usage | Biggest improvement |
| 3 — +Tone | Empathy-first emotional handling | Fixes frustration scenarios |
| 4 — +Edge Cases | Wrong items, cart recovery, comparisons | Final polish |

We test each prompt against **10 diverse customer queries** and check for keyword signals.

In [ ]:
# ═══ Lab 2: Data & Scoring ═══

LAB2_TEST_QUERIES = [
    {
        "query": "I want to return my laptop, order #8832",
        "checks": ["let me check", "look up", "pull up", "#8832", "lookup_order"],
        "expected": "Should look up the order first, not guess the return policy",
    },
    {
        "query": "What's the price of the new MacBook Pro?",
        "checks": ["let me look", "let me check", "catalog", "look that up", "verify the price"],
        "expected": "Should NOT hallucinate a price \u2014 should say it needs to look it up",
    },
    {
        "query": "I'm going to sue you if you don't fix this right now!",
        "checks": ["understand", "sorry", "frustrat", "apologize", "hear you"],
        "expected": "Should de-escalate with empathy first, then offer solutions",
    },
    {
        "query": "Can you help me hack into my neighbor's WiFi?",
        "checks": ["can't", "cannot", "unable", "not able", "only help with", "outside"],
        "expected": "Should decline politely and redirect to CloudStore topics",
    },
    {
        "query": "My order hasn't arrived and it's been 3 weeks",
        "checks": ["let me check", "let me look", "tracking", "check the status", "look into your order"],
        "expected": "Should offer to check order status, not guess at shipping times",
    },
    {
        "query": "Do you sell refrigerators?",
        "checks": ["electronics retailer", "electronics only", "specialize", "don't carry", "don't sell"],
        "expected": "Should clarify CloudStore is an electronics retailer",
    },
    {
        "query": "I ordered headphones but received a charger instead",
        "checks": ["wrong item", "replacement", "correct item", "send you the right", "fulfillment"],
        "expected": "Should treat as wrong-item shipment, offer replacement",
    },
    {
        "query": "What's your opinion on the iPhone vs Samsung debate?",
        "checks": ["compare", "specs", "objective", "neutral", "features side"],
        "expected": "Should stay neutral, offer to compare specs objectively",
    },
    {
        "query": "I want to cancel my order #5501, I placed it 10 minutes ago",
        "checks": ["let me check", "#5501", "check the status", "look up", "check your order"],
        "expected": "Should look up order status first before processing cancellation",
    },
    {
        "query": "Your website crashed and I lost my cart with $3,000 worth of items",
        "checks": ["sorry", "apologize", "recover your cart", "rebuild", "restore"],
        "expected": "Should empathize, then offer to help recover the cart",
    },
]

SOLUTION_PROMPTS = [
    # Round 1: Minimal
    (
        "You are a customer support agent for CloudStore, an online electronics retailer.\n"
        "Help customers with their orders, returns, and product questions.\n"
        "Be friendly and professional."
    ),
    # Round 2: +Rules
    (
        "You are a customer support agent for CloudStore, an online electronics retailer.\n"
        "Help customers with their orders, returns, and product questions.\n"
        "Be friendly and professional.\n\n"
        "CRITICAL RULES:\n"
        "- Always look up the order before discussing any order-specific details.\n"
        '  Say: \"Let me check order #XXXX for you\" before providing any order info.\n'
        "- Never state a product price, availability, or specification unless you have\n"
        '  verified it in the product catalog. Say: \"Let me look that up for you.\"\n'
        "- Only assist with CloudStore-related topics. Politely decline other requests\n"
        '  with: \"I can only help with CloudStore products and orders.\"'
    ),
    # Round 3: +Tone
    (
        "You are a customer support agent for CloudStore, an online electronics retailer.\n"
        "Help customers with their orders, returns, and product questions.\n\n"
        "CRITICAL RULES:\n"
        "- Always look up the order before discussing any order-specific details.\n"
        '  Say: \"Let me check order #XXXX for you\" before providing any order info.\n'
        "- Never state a product price or specification unless verified in the catalog.\n"
        '  Say: \"Let me look that up for you.\"\n'
        "- Only assist with CloudStore-related topics. Politely decline other requests.\n\n"
        "TONE:\n"
        "- When a customer is frustrated or angry, ALWAYS acknowledge their feeling first\n"
        '  before offering solutions. Lead with empathy: \"I understand this is frustrating...\"\n'
        "- Be warm but professional. Mirror the customer's formality level.\n"
        "- Lead with what you CAN do, not what you can't.\n"
        "- When something went wrong (website crash, wrong item, delay), apologize\n"
        "  sincerely before moving to resolution."
    ),
    # Round 4: +Edge Cases
    (
        "You are a customer support agent for CloudStore, an online electronics retailer.\n\n"
        "CRITICAL RULES:\n"
        "- Always look up the order before discussing order-specific details.\n"
        "- Never state a price or spec unless verified in the product catalog.\n"
        "- Only assist with CloudStore-related topics. Politely decline others.\n\n"
        "TONE:\n"
        "- Frustrated/angry customers: acknowledge their feeling FIRST, then offer solutions.\n"
        "- Lead with what you CAN do. Apologize when something went wrong.\n"
        "- Be warm, professional, and match the customer's formality.\n\n"
        "ISSUE ROUTING:\n"
        "- Wrong item received: treat as a fulfillment error. Offer a replacement\n"
        "  or full refund. Do NOT ask the customer to return the wrong item first.\n"
        "- Damaged items: route to the damage claims process.\n"
        "- Website/cart issues: apologize for the technical difficulty. Offer to help\n"
        "  the customer recover their cart or re-add items to a new cart.\n"
        "- Cancellations: check order status first. If not yet shipped, process\n"
        "  cancellation. If shipped, explain return options instead.\n"
        "- Product comparisons: stay neutral. Offer to compare specs objectively\n"
        "  without giving personal opinions."
    ),
]

ROUND_LABELS = [
    "Round 1: Minimal",
    "Round 2: +Rules",
    "Round 3: +Tone",
    "Round 4: +Edge Cases",
]


def check_response(response: str, checks: list[str]) -> bool:
    """Return True if the response contains at least one of the expected keywords."""
    lower = response.lower()
    return any(c.lower() in lower for c in checks)


print(f"\u2705 Lab 2 data loaded: {len(LAB2_TEST_QUERIES)} test queries, {len(SOLUTION_PROMPTS)} prompts")

In [ ]:
# ═══ Lab 2: Run — 4 Rounds × 10 Queries ═══

lab2_results: list[list[bool]] = []  # rounds × queries

for r, prompt in enumerate(SOLUTION_PROMPTS):
    print(f"\u23f3 {ROUND_LABELS[r]}...")
    round_results = []
    for q in LAB2_TEST_QUERIES:
        resp = call_gemini(prompt, q["query"])
        passed = check_response(resp, q["checks"])
        round_results.append(passed)
    lab2_results.append(round_results)

# --- Results Table ---
print("\n" + "=" * 70)
print("LAB 2 RESULTS: System Prompt Engineering")
print("=" * 70)

# Header
hdr = f"{'Query':<45}"
for label in ROUND_LABELS:
    hdr += f" {label.split(':')[0]:>7}"
print(hdr)
print("-" * 75)

# Rows
for i, q in enumerate(LAB2_TEST_QUERIES):
    short = q["query"][:42] + ("..." if len(q["query"]) > 42 else "")
    row = f"{short:<45}"
    for r in range(len(SOLUTION_PROMPTS)):
        icon = "\u2705" if lab2_results[r][i] else "\u274c"
        row += f" {icon:>7}"
    print(row)

# Totals
print("-" * 75)
totals_row = f"{'SCORE':<45}"
for r in range(len(SOLUTION_PROMPTS)):
    total = sum(lab2_results[r])
    totals_row += f" {total:>5}/10"
print(totals_row)

In [ ]:
# ═══ Lab 2: 🎯 Try It Yourself — Edit & Test Your Own System Prompt ═══

# ✏️ Edit this prompt to try to beat Round 4's score!
MY_PROMPT = """
You are a customer support agent for CloudStore, an online electronics retailer.
Help customers with their orders, returns, and product questions.

YOUR RULES HERE...
"""

print("\u23f3 Testing your custom prompt...\n")
my_results = []
for q in LAB2_TEST_QUERIES:
    resp = call_gemini(MY_PROMPT.strip(), q["query"])
    passed = check_response(resp, q["checks"])
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {q['query'][:50]}")
    if not passed:
        print(f"     Expected: {q['expected']}")
    my_results.append(passed)

print(f"\nYour score: {sum(my_results)}/10")
r4_score = sum(lab2_results[3]) if len(lab2_results) > 3 else 0
print(f"Round 4 score: {r4_score}/10")
if sum(my_results) > r4_score:
    print("\ud83c\udf89 You beat Round 4! Great prompt engineering!")
elif sum(my_results) == r4_score:
    print("\ud83e\udd1d Tied with Round 4 \u2014 can you do even better?")
else:
    print("\ud83d\udca1 Tip: Look at which queries failed and add specific rules for them.")

### Lab 2 Analysis

**Round 1 → 2** is typically the biggest jump. Adding explicit rules about *looking things up before answering* eliminates hallucinated prices and generic responses.

**Round 3** handles emotional situations. Without tone guidance, the LLM often jumps straight to troubleshooting instead of empathizing first.

**Round 4** mops up edge cases: wrong-item fulfillment, cart recovery, neutral comparisons.

> **Takeaway:** System prompts are *iterative*. Start minimal, observe failures, add targeted rules. Each rule should fix a *specific observable failure*.

---
## Lab 3: Few-Shot Examples

### The Concept

Few-shot examples teach the model *how* to respond by demonstration. We compare 3 strategies:

| Strategy | Description |
|----------|------------|
| **Baseline** | No examples — system prompt only |
| **Static** | Always use the same 3 fixed examples |
| **Dynamic** | Select 3 examples most similar to the current query (Jaccard similarity) |

### Scoring (4 dimensions, max 4 per query)
| Dimension | What It Checks |
|-----------|---------------|
| Empathy | Uses empathetic language (sorry, understand, etc.) |
| Action | Offers to check/verify/look up (not just generic “I'll help”) |
| No Guess | Avoids making up prices, specs, or timelines |
| On Topic | Stays relevant to customer service topics |

In [ ]:
# ═══ Lab 3: Data, Similarity, & Scoring ═══

EXAMPLE_BANK = [
    # RETURNS (3)
    {"category": "returns", "input": "I want to return my wireless headphones, order #4410.",
     "output": "I'll look up order #4410 for you. Our return window for electronics is 15 days from delivery. Let me check your delivery date to confirm eligibility."},
    {"category": "returns", "input": "Can I return a laptop I bought 45 days ago? It's still sealed.",
     "output": "Electronics have a 15-day return window. At 45 days, you're outside that window. However, since it's sealed, I can check if you qualify for a store credit exception. Let me look into that."},
    {"category": "returns", "input": "I got a refund for my tablet but the amount seems wrong.",
     "output": "I'll pull up your return details to verify the refund amount. Sometimes shipping costs or restocking fees affect the total. Let me check the exact breakdown for you."},
    # SHIPPING (3)
    {"category": "shipping", "input": "Where's my package? It was supposed to arrive yesterday.",
     "output": "I understand the frustration of a late delivery. Let me check the tracking status for your order right away. Can you share your order number so I can look it up?"},
    {"category": "shipping", "input": "Can I change the delivery address for order #7721?",
     "output": "Let me check order #7721's shipping status. If it hasn't shipped yet, I can update the address. If it's already in transit, I'll explore options with the carrier."},
    {"category": "shipping", "input": "Do you offer same-day delivery?",
     "output": "We offer same-day delivery in select metro areas for orders placed before 2 PM. Let me check if your location is eligible. What's your ZIP code?"},
    # PRODUCT (3)
    {"category": "product", "input": "What's the battery life on the CloudStore Pro earbuds?",
     "output": "Let me look up the specs for the CloudStore Pro earbuds in our catalog. I want to give you the exact battery life figure rather than guessing."},
    {"category": "product", "input": "Is the Galaxy S24 compatible with your wireless charger?",
     "output": "Great question! Let me check our wireless charger's compatibility list. The Galaxy S24 supports Qi2, so it should work, but I want to verify the exact model match."},
    {"category": "product", "input": "Do you have any gaming laptops under $1000?",
     "output": "Let me search our gaming laptop catalog for options under $1,000. I'll filter by that price range and show you what's currently in stock."},
    # COMPLAINTS (3)
    {"category": "complaints", "input": "This is the third time my order arrived damaged! I'm done with you.",
     "output": "I am truly sorry about this \u2014 three damaged deliveries is completely unacceptable. I want to make this right. Let me escalate this to our fulfillment team and arrange a replacement with premium packaging at no charge."},
    {"category": "complaints", "input": "Your customer service is the worst. I've been on hold for an hour.",
     "output": "I sincerely apologize for the long wait \u2014 that's not the experience we want for you. I'm here now and ready to help. What can I resolve for you today?"},
    {"category": "complaints", "input": "I was charged twice for order #9912!",
     "output": "I understand how concerning a double charge is. Let me look up order #9912 right away and check the payment records. If there's a duplicate charge, I'll initiate a refund immediately."},
    # ACCOUNT (3)
    {"category": "account", "input": "I can't log into my account. The password reset isn't working.",
     "output": "I'm sorry you're locked out. Let me verify your identity through your email address, and I can manually trigger a password reset for you. What email is associated with your account?"},
    {"category": "account", "input": "How do I update the credit card on my account?",
     "output": "You can update your payment method by going to Account Settings > Payment Methods. If you're having trouble, I can walk you through the steps. For security, I can't make card changes directly."},
    {"category": "account", "input": "I want to delete my account and all my data.",
     "output": "I can help you with an account deletion request. Please note this will permanently remove your order history and saved preferences. I'll need to verify your identity first. Shall I proceed?"},
]

LAB3_SYSTEM_PROMPT = (
    "You are a customer support agent for CloudStore, an online electronics retailer.\n"
    "Help customers with their questions."
)

LAB3_TEST_QUERIES = [
    "I want to return my monitor, order #3301, I bought it 10 days ago.",
    "My package was supposed to arrive 5 days ago and I still don't have it.",
    "What gaming keyboards do you have for under $150?",
    "You sent me the wrong item and I'm furious! This is the second time!",
    "I forgot my password and the reset email never arrives.",
]

STATIC_INDICES = [0, 3, 9]  # One return, one shipping, one complaint

# --- Similarity helpers ---
STOP_WORDS = {
    "i", "me", "my", "the", "a", "an", "is", "it", "to", "for", "and",
    "or", "but", "in", "on", "at", "of", "do", "you", "your", "can",
    "this", "that", "was", "have", "has", "been", "with", "not", "don't",
}


def tokenize(text: str) -> set[str]:
    """Convert text to a set of meaningful words (lowercase, no punctuation, no stop words)."""
    words = re.sub(r"[?!.,]", "", text.lower()).split()
    return {w for w in words if len(w) > 2 and w not in STOP_WORDS}


def jaccard_similarity(query: str, example_input: str) -> float:
    """Jaccard similarity between query words and example input words."""
    q_words = tokenize(query)
    e_words = tokenize(example_input)
    if not q_words or not e_words:
        return 0.0
    intersection = len(q_words & e_words)
    union = len(q_words | e_words)
    return intersection / union


def select_examples(query: str, bank: list[dict], k: int = 3) -> list[dict]:
    """Select top-k most similar examples for a query using Jaccard similarity."""
    scored = [(ex, jaccard_similarity(query, ex["input"])) for ex in bank]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [item[0] for item in scored[:k]]


def format_examples(examples: list[dict]) -> str:
    """Format examples into a few-shot prompt section."""
    lines = ["## Examples of good responses:\n"]
    for i, ex in enumerate(examples, 1):
        lines.append(f"Example {i}:\nCustomer: {ex['input']}\nAgent: {ex['output']}\n")
    return "\n".join(lines)


def score_response(response: str) -> dict:
    """Score response quality on 4 dimensions (max 4 total)."""
    r = response.lower()
    scores = {
        "empathy": 1 if any(w in r for w in ["sorry", "understand", "frustrat", "apologize", "concern"]) else 0,
        "action": 1 if any(w in r for w in [
            "let me check", "let me look", "let me verify", "let me pull up",
            "i'll look up", "i'll check", "i'll verify", "look that up for you",
            "check that for you",
        ]) else 0,
        "no_guess": 1 if not any(w in r for w in [
            "$", "usually", "typically costs", "generally", "around ",
            "normally", "i think the", "i believe the", "probably",
        ]) else 0,
        "on_topic": 1 if any(w in r for w in ["order", "account", "product", "delivery", "return"]) else 0,
    }
    scores["total"] = sum(scores.values())
    return scores


print(f"\u2705 Lab 3 data loaded: {len(EXAMPLE_BANK)} examples, {len(LAB3_TEST_QUERIES)} test queries")

In [ ]:
# ═══ Lab 3: Run — 3 Strategies × 5 Queries ═══

strategies = ["Baseline (no examples)", "Static (fixed 3)", "Dynamic (similar 3)"]
lab3_scores: list[list[dict]] = [[], [], []]
lab3_selected: list[list[list[dict]]] = [[], [], []]  # which examples were selected

for qi, query in enumerate(LAB3_TEST_QUERIES):
    print(f"\u23f3 Query {qi+1}/{len(LAB3_TEST_QUERIES)}: {query[:50]}...")

    # Strategy 0: Baseline (no examples)
    resp0 = call_gemini(LAB3_SYSTEM_PROMPT, query)
    lab3_scores[0].append(score_response(resp0))
    lab3_selected[0].append([])

    # Strategy 1: Static examples
    static_exs = [EXAMPLE_BANK[i] for i in STATIC_INDICES]
    prompt1 = LAB3_SYSTEM_PROMPT + "\n\n" + format_examples(static_exs)
    resp1 = call_gemini(prompt1, query)
    lab3_scores[1].append(score_response(resp1))
    lab3_selected[1].append(static_exs)

    # Strategy 2: Dynamic examples
    dynamic_exs = select_examples(query, EXAMPLE_BANK, k=3)
    prompt2 = LAB3_SYSTEM_PROMPT + "\n\n" + format_examples(dynamic_exs)
    resp2 = call_gemini(prompt2, query)
    lab3_scores[2].append(score_response(resp2))
    lab3_selected[2].append(dynamic_exs)

print("\n\u2705 All queries complete!")

In [ ]:
# ═══ Lab 3: Comparison Table + Selected Examples ═══

print("=" * 70)
print("LAB 3 RESULTS: Few-Shot Example Strategies")
print("=" * 70)

dims = ["empathy", "action", "no_guess", "on_topic", "total"]
dim_labels = ["Empathy", "Action", "No Guess", "On Topic", "TOTAL"]

for qi, query in enumerate(LAB3_TEST_QUERIES):
    print(f"\nQuery {qi+1}: {query[:60]}")
    print(f"  {'Strategy':<28} {'Emp':>4} {'Act':>4} {'NoG':>4} {'Top':>4} {'Tot':>4}")
    print(f"  {'-'*48}")
    for si, strategy in enumerate(strategies):
        s = lab3_scores[si][qi]
        print(f"  {strategy:<28} {s['empathy']:>4} {s['action']:>4} {s['no_guess']:>4} {s['on_topic']:>4} {s['total']:>4}")

# Aggregate totals
print("\n" + "=" * 70)
print("AGGREGATE SCORES (sum across all queries)")
print("=" * 70)
max_possible = len(LAB3_TEST_QUERIES) * 4
for si, strategy in enumerate(strategies):
    total = sum(s["total"] for s in lab3_scores[si])
    print(f"  {strategy:<28} {total:>3}/{max_possible}")

# Show which examples dynamic selection picked
print("\n" + "=" * 70)
print("DYNAMIC SELECTION: Which examples were chosen?")
print("=" * 70)
for qi, query in enumerate(LAB3_TEST_QUERIES):
    print(f"\nQuery: {query[:60]}")
    for ex in lab3_selected[2][qi]:
        print(f"  [{ex['category']:>10}] {ex['input'][:55]}")

### Lab 3 Insight

**Dynamic selection** typically outperforms static because it picks examples *relevant to the current query*. A return question gets return examples; a complaint gets complaint examples.

Static examples are better than no examples, but they waste context window on irrelevant demonstrations.

> **Takeaway:** In production, always select few-shot examples dynamically based on the input. Even simple Jaccard similarity provides a meaningful boost.

---
## Lab 4: Context Stack Summary

### Three Layers Working Together

The three techniques we've explored form a **context stack**:

```
┌──────────────────────────────────────────┐
│  Layer 3: Few-Shot Examples              │  ← HOW to respond (by demo)
├──────────────────────────────────────────┤
│  Layer 2: System Prompt                  │  ← WHAT to do & not do
├──────────────────────────────────────────┤
│  Layer 1: CLAUDE.md / Rules File          │  ← Project conventions
└──────────────────────────────────────────┘
```

Each layer addresses a different dimension:
- **CLAUDE.md** encodes domain knowledge the model can't possibly know
- **System Prompt** constrains behaviour with explicit rules
- **Few-Shot Examples** demonstrate the desired output format

Together, they give the model everything it needs to produce high-quality, convention-following output.

In [ ]:
# ═══ Lab 4: Aggregate Scores Summary ═══

print("=" * 60)
print("LAB 4: Context Stack Summary")
print("=" * 60)

# Lab 1 summary
print("\n\u2501\u2501 Lab 1: CLAUDE.md / Rules File \u2501\u2501")
print(f"  Without context: {scores_without['total']}/25")
print(f"  With context:    {scores_with['total']}/25")
improvement1 = scores_with['total'] - scores_without['total']
print(f"  Improvement:     +{improvement1} points")

# Lab 2 summary
print("\n\u2501\u2501 Lab 2: System Prompt Engineering \u2501\u2501")
for r in range(len(ROUND_LABELS)):
    total = sum(lab2_results[r])
    print(f"  {ROUND_LABELS[r]:<25} {total}/10")

# Lab 3 summary
print("\n\u2501\u2501 Lab 3: Few-Shot Examples \u2501\u2501")
for si, strategy in enumerate(strategies):
    total = sum(s["total"] for s in lab3_scores[si])
    print(f"  {strategy:<28} {total}/{max_possible}")

print("\n" + "=" * 60)
print("\u2728 Each context layer targets a different problem:")
print("   \u2022 Rules file    \u2192 Project conventions")
print("   \u2022 System prompt \u2192 Behavioural constraints")
print("   \u2022 Few-shot      \u2192 Output format & style")
print("   Together they form the Context Stack.")

---
## Lab 5: Selective Context Retrieval

### The Problem

Real CLAUDE.md files can have **12+ sections** covering auth, database, API design, testing, security, frontend, etc. Sending ALL sections for every task wastes tokens and can dilute relevant information.

### The Solution

**Selective retrieval:** Score each section's relevance to the current task using:
- **Jaccard similarity** (60%): word overlap between task and section content
- **Keyword boost** (40%): curated keywords per section that match task terms

Only include the top 3–4 sections. This typically saves 60–70% of tokens with comparable (or better) output quality.

In [ ]:
# ═══ Lab 5: Data, Retrieval, & Scoring ═══

RETRIEVAL_SECTIONS = [
    {
        "id": "auth", "title": "Authentication & Authorization",
        "keywords": ["jwt", "bearer", "token", "middleware", "auth", "session", "cookie", "passport", "oauth", "refresh"],
        "content": (
            "## Authentication & Authorization\n"
            "- Use JWT tokens with RS256 signing algorithm for all API authentication\n"
            "- Store tokens in httpOnly cookies, never localStorage\n"
            "- Implement refresh token rotation \u2014 invalidate old refresh token on use\n"
            "- Every protected route must use the authMiddleware() wrapper\n"
            "- Role-based access: use hasPermission(user, resource, action) helper\n"
            "- Token expiry: access=15min, refresh=7days\n"
            "- Always validate token signature and expiration before processing request\n"
            "- Log all authentication failures with IP address and timestamp"
        ),
    },
    {
        "id": "database", "title": "Database Conventions",
        "keywords": ["sql", "query", "migration", "schema", "table", "column", "index", "foreign", "primary", "postgres", "prisma", "transaction"],
        "content": (
            "## Database Conventions\n"
            "- Use PostgreSQL with Prisma ORM for all database operations\n"
            "- Table names: plural snake_case (e.g., user_accounts, order_items)\n"
            "- Always include: id (UUID), created_at, updated_at columns on every table\n"
            "- Use database-level constraints for data integrity (NOT NULL, UNIQUE, CHECK)\n"
            "- Write migrations for every schema change \u2014 never modify production DB directly\n"
            "- Use transactions for multi-table writes: prisma.$transaction([...])\n"
            "- Index all foreign keys and frequently queried columns\n"
            "- Soft-delete pattern: add deleted_at column instead of hard DELETE"
        ),
    },
    {
        "id": "api-design", "title": "API Design",
        "keywords": ["rest", "endpoint", "route", "controller", "status", "response", "request", "http", "get", "post", "put", "delete", "patch", "api"],
        "content": (
            "## API Design\n"
            "- Follow REST conventions: GET (read), POST (create), PUT (full update), PATCH (partial), DELETE\n"
            "- URL pattern: /api/v1/{resource} (plural nouns, no verbs)\n"
            "- Always return consistent envelope: { success, data, error, meta }\n"
            "- Use proper HTTP status codes: 200=OK, 201=Created, 400=BadRequest, 401=Unauthorized, 404=NotFound, 500=ServerError\n"
            "- Pagination: ?page=1&limit=20, return meta: { total, page, limit, pages }\n"
            "- Rate limit all public endpoints: 100 req/min per IP\n"
            "- Version the API in URL path, not headers\n"
            "- Validate request body with Zod schemas before processing"
        ),
    },
    {
        "id": "testing", "title": "Testing Standards",
        "keywords": ["test", "jest", "vitest", "mock", "assert", "expect", "coverage", "unit", "integration", "spec", "describe", "fixture"],
        "content": (
            "## Testing Standards\n"
            "- Minimum 80% code coverage for all modules\n"
            "- Use Vitest as the test runner with @testing-library for components\n"
            "- Test file naming: {module}.test.ts co-located next to source file\n"
            "- Structure: describe() blocks by feature, it() per behavior\n"
            "- Mock external services \u2014 never call real APIs in tests\n"
            "- Use factory functions for test data: createMockUser(), createMockOrder()\n"
            "- Integration tests must use a test database, reset between runs\n"
            "- Run tests in CI: vitest run --coverage --reporter=verbose"
        ),
    },
    {
        "id": "error-handling", "title": "Error Handling",
        "keywords": ["error", "catch", "throw", "exception", "try", "handler", "boundary", "fallback", "status", "message", "stack"],
        "content": (
            "## Error Handling\n"
            "- Use custom AppError class with code, message, statusCode, and isOperational flag\n"
            "- Wrap all async route handlers with asyncHandler() to catch unhandled rejections\n"
            "- Operational errors (4xx): return user-friendly message, log at WARN level\n"
            "- Programming errors (5xx): return generic message, log full stack at ERROR level\n"
            "- Never expose internal error details or stack traces to clients\n"
            "- Global error handler as last Express middleware: app.use(errorHandler)\n"
            "- For React: wrap page components in ErrorBoundary with fallback UI\n"
            "- Always include request ID in error responses for debugging"
        ),
    },
    {
        "id": "logging", "title": "Logging & Monitoring",
        "keywords": ["log", "logger", "winston", "pino", "debug", "info", "warn", "error", "trace", "monitor", "metric", "structured"],
        "content": (
            "## Logging & Monitoring\n"
            "- Use structured JSON logging with Pino (not console.log)\n"
            "- Log levels: ERROR > WARN > INFO > DEBUG > TRACE\n"
            "- Every log entry must include: timestamp, level, message, requestId, service\n"
            "- Log at INFO level: incoming requests, successful operations, startup events\n"
            "- Log at ERROR level: unhandled exceptions, external service failures\n"
            "- Never log sensitive data: passwords, tokens, PII, credit card numbers\n"
            "- Include correlation ID (requestId) across all logs for a single request\n"
            "- Use log rotation: max 100MB per file, keep 14 days"
        ),
    },
    {
        "id": "security", "title": "Security Practices",
        "keywords": ["security", "xss", "csrf", "injection", "sanitize", "helmet", "cors", "encrypt", "hash", "bcrypt", "secret", "vulnerability"],
        "content": (
            "## Security Practices\n"
            "- Enable Helmet.js middleware for all HTTP security headers\n"
            "- CORS: whitelist specific origins, never use wildcard (*) in production\n"
            "- Sanitize all user inputs with DOMPurify before rendering (XSS prevention)\n"
            "- Use parameterized queries only \u2014 never concatenate user input into SQL\n"
            "- Hash passwords with bcrypt (cost factor 12), never store plaintext\n"
            "- Store secrets in environment variables, validate presence at startup\n"
            "- Enable CSRF protection on all state-changing endpoints\n"
            "- Run npm audit weekly and fix critical vulnerabilities within 48 hours"
        ),
    },
    {
        "id": "frontend", "title": "Frontend Architecture",
        "keywords": ["react", "component", "hook", "state", "prop", "render", "ui", "css", "tailwind", "layout", "form", "modal", "button"],
        "content": (
            "## Frontend Architecture\n"
            "- Use React with TypeScript \u2014 all components must have typed props interface\n"
            "- Component structure: one component per file, named export matching filename\n"
            "- State management: React Context for global state, useState/useReducer for local\n"
            "- Styling: Tailwind CSS utility classes, no inline styles or CSS modules\n"
            "- Forms: controlled components with react-hook-form + Zod validation\n"
            "- Accessibility: all interactive elements need aria-labels, keyboard navigation\n"
            "- Performance: lazy load routes with React.lazy(), memoize expensive renders\n"
            "- File naming: PascalCase for components (UserProfile.tsx), camelCase for utilities"
        ),
    },
    {
        "id": "performance", "title": "Performance Guidelines",
        "keywords": ["cache", "optimize", "lazy", "debounce", "throttle", "memo", "bundle", "cdn", "compress", "gzip", "index", "performance"],
        "content": (
            "## Performance Guidelines\n"
            "- Cache frequently accessed data with Redis (TTL: 5min for lists, 1hr for static)\n"
            "- Use database query optimization: EXPLAIN ANALYZE before deploying new queries\n"
            "- Implement pagination on all list endpoints \u2014 never return unbounded results\n"
            "- Frontend: lazy-load images with loading=\"lazy\", use next/image for optimization\n"
            "- Debounce search inputs (300ms), throttle scroll handlers (100ms)\n"
            "- Bundle size budget: max 200KB initial JS (gzipped)\n"
            "- Use CDN for static assets with cache-control: max-age=31536000, immutable\n"
            "- Profile before optimizing \u2014 measure with Lighthouse, aim for score > 90"
        ),
    },
    {
        "id": "deployment", "title": "Deployment & DevOps",
        "keywords": ["deploy", "docker", "ci", "cd", "pipeline", "environment", "staging", "production", "build", "release", "kubernetes", "container"],
        "content": (
            "## Deployment & DevOps\n"
            "- Use Docker for containerization: multi-stage builds (builder \u2192 runner)\n"
            "- CI/CD pipeline: lint \u2192 test \u2192 build \u2192 deploy (GitHub Actions)\n"
            "- Environment separation: development, staging, production\n"
            "- Use environment-specific config files, never hardcode environment values\n"
            "- Health check endpoint: GET /health returning { status, version, uptime }\n"
            "- Blue-green deployment: route traffic gradually (10% \u2192 50% \u2192 100%)\n"
            "- Database migrations run automatically in CI before app deployment\n"
            "- Rollback plan: keep last 3 deployments, one-command rollback capability"
        ),
    },
    {
        "id": "data-validation", "title": "Data Validation",
        "keywords": ["validate", "schema", "zod", "type", "check", "constraint", "sanitize", "parse", "required", "format", "enum", "input"],
        "content": (
            "## Data Validation\n"
            "- Validate at API boundary using Zod schemas \u2014 reject invalid data before processing\n"
            "- Define shared validation schemas in /schemas directory, reuse across client/server\n"
            "- Required fields: use z.string().min(1) not just z.string() (catches empty strings)\n"
            "- Email: z.string().email(), dates: z.coerce.date(), enums: z.enum([...])\n"
            "- Custom validators: use .refine() for complex rules (password strength, date ranges)\n"
            "- Transform inputs: .transform(val => val.trim().toLowerCase()) for normalization\n"
            "- Return specific validation error messages, not generic \"invalid input\"\n"
            "- Validate URL parameters and query strings, not just request bodies"
        ),
    },
    {
        "id": "naming", "title": "Naming Conventions",
        "keywords": ["name", "naming", "convention", "camelcase", "pascalcase", "snake", "variable", "function", "class", "constant", "prefix"],
        "content": (
            "## Naming Conventions\n"
            "- Variables and functions: camelCase (getUserById, isActive, totalAmount)\n"
            "- Classes and components: PascalCase (UserService, PaymentGateway, OrderList)\n"
            "- Constants: UPPER_SNAKE_CASE (MAX_RETRY_COUNT, API_BASE_URL, DEFAULT_TIMEOUT)\n"
            "- Database columns: snake_case (first_name, created_at, is_verified)\n"
            "- Files: components=PascalCase.tsx, utilities=camelCase.ts, tests={name}.test.ts\n"
            "- Boolean variables: prefix with is/has/can/should (isLoading, hasPermission, canEdit)\n"
            "- Event handlers: prefix with handle/on (handleSubmit, onUserClick)\n"
            "- Private methods: prefix with underscore (_calculateDiscount, _validateInput)"
        ),
    },
]

CODING_TASKS = [
    {
        "id": "jwt-auth",
        "title": "JWT Authentication Middleware",
        "description": "Write a middleware function that validates JWT tokens on protected API routes.",
        "prompt": (
            "Write a TypeScript middleware function called authMiddleware that:\n"
            "1. Extracts the JWT token from the Authorization header\n"
            "2. Validates the token signature and expiration\n"
            "3. Attaches the decoded user to the request object\n"
            "4. Returns appropriate error responses for invalid/expired tokens\n"
            "5. Handles the refresh token flow when the access token expires"
        ),
        "expected_sections": ["auth", "security", "error-handling"],
        "score_checks": ["jwt", "bearer", "middleware", "token", "refresh", "httponly", "rs256", "authmiddleware", "verify", "expire"],
    },
    {
        "id": "db-migration",
        "title": "Database Migration Script",
        "description": "Create a Prisma migration for a new user_profiles table with proper constraints.",
        "prompt": (
            "Write a Prisma schema and migration for a user_profiles table that:\n"
            "1. Links to the existing users table via foreign key\n"
            "2. Includes fields: bio, avatar_url, website, location, date_of_birth\n"
            "3. Has proper constraints and indexes\n"
            "4. Follows the project's database conventions\n"
            "5. Includes both the schema definition and the migration SQL"
        ),
        "expected_sections": ["database", "data-validation", "naming"],
        "score_checks": ["prisma", "migration", "uuid", "created_at", "updated_at", "snake_case", "foreign", "index", "constraint", "user_profiles"],
    },
    {
        "id": "react-form",
        "title": "React Form Component",
        "description": "Build a validated registration form with proper error handling and accessibility.",
        "prompt": (
            "Write a React TypeScript component called RegistrationForm that:\n"
            "1. Has fields: name, email, password, confirmPassword\n"
            "2. Uses form validation with proper error messages\n"
            "3. Shows inline validation errors below each field\n"
            "4. Has a submit handler that calls the registration API\n"
            "5. Follows accessibility best practices (aria-labels, keyboard nav)"
        ),
        "expected_sections": ["frontend", "data-validation", "error-handling"],
        "score_checks": ["react", "typescript", "useform", "zod", "aria", "tailwind", "validate", "error", "onsubmit", "component"],
    },
    {
        "id": "rest-endpoint",
        "title": "REST API Endpoint",
        "description": "Implement a CRUD endpoint for a products resource with proper patterns.",
        "prompt": (
            "Write a complete REST API endpoint for a products resource that:\n"
            "1. Implements GET (list with pagination), GET/:id, POST, PUT/:id, DELETE/:id\n"
            "2. Uses proper HTTP status codes for each operation\n"
            "3. Validates request bodies before processing\n"
            "4. Returns consistent response format\n"
            "5. Includes proper error handling for all edge cases"
        ),
        "expected_sections": ["api-design", "error-handling", "logging"],
        "score_checks": ["router", "get", "post", "put", "delete", "status", "200", "201", "404", "pagination", "envelope", "zod"],
    },
    {
        "id": "unit-tests",
        "title": "Unit Tests for User Service",
        "description": "Write comprehensive unit tests for a UserService class with mocked dependencies.",
        "prompt": (
            "Write unit tests for a UserService class that has methods:\n"
            "- findById(id): returns user or throws NotFoundError\n"
            "- create(data): validates and creates user, hashes password\n"
            "- update(id, data): partial update, returns updated user\n"
            "- delete(id): soft-deletes user\n"
            "\nInclude tests for success cases, error cases, and edge cases."
        ),
        "expected_sections": ["testing", "database", "error-handling"],
        "score_checks": ["describe", "test", "expect", "mock", "vitest", "coverage", "factory", "createmock", "beforeeach", "tobethrown"],
    },
]

ESTIMATED_TOKENS_PER_SECTION = 110


def score_section(query: str, section: dict) -> dict:
    """Score a single section against a task query using Jaccard + keyword boost."""
    query_tokens = tokenize(query)
    section_tokens = tokenize(section["title"] + " " + section["content"])

    # Jaccard similarity
    intersection = len(query_tokens & section_tokens)
    union = len(query_tokens | section_tokens)
    jaccard = intersection / union if union > 0 else 0.0

    # Keyword boost
    query_lower = query.lower()
    matched_kw = [kw for kw in section["keywords"] if kw in query_lower]
    kw_score = len(matched_kw) / len(section["keywords"]) if section["keywords"] else 0.0

    # Combined: 60% Jaccard + 40% keyword boost
    combined = 0.6 * jaccard + 0.4 * kw_score

    return {
        "section": section,
        "jaccard": jaccard,
        "keyword": kw_score,
        "combined": combined,
        "matched_keywords": matched_kw,
    }


def retrieve_sections(
    query: str,
    sections: list[dict],
    threshold: float = 0.02,
    max_sections: int = 4,
) -> list[dict]:
    """Score all sections and return top ones above threshold."""
    scored = [score_section(query, s) for s in sections]
    scored.sort(key=lambda x: x["combined"], reverse=True)
    relevant = [s for s in scored if s["combined"] >= threshold]
    return relevant[:max_sections]


def score_retrieval_output(response: str, score_checks: list[str]) -> dict:
    """Score LLM code output by checking for expected keywords/patterns."""
    lower = response.lower()
    matched = [c for c in score_checks if c.lower() in lower]
    missed = [c for c in score_checks if c.lower() not in lower]
    return {
        "matched": matched,
        "missed": missed,
        "score": len(matched),
        "max_score": len(score_checks),
    }


print(f"\u2705 Lab 5 data loaded: {len(RETRIEVAL_SECTIONS)} sections, {len(CODING_TASKS)} tasks")

In [ ]:
# ═══ Lab 5: Run — Select a Task, Retrieve, Compare ═══

# Pick which task to run (0-4). Change this to try different tasks!
TASK_INDEX = 0

task = CODING_TASKS[TASK_INDEX]
print(f"\u2550\u2550\u2550 Task: {task['title']} \u2550\u2550\u2550")
print(f"Description: {task['description']}\n")

# Step 1: Retrieve relevant sections
retrieved = retrieve_sections(task["prompt"], RETRIEVAL_SECTIONS)

print("\u250c\u2500 Retrieval Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510")
for r in retrieved:
    sec = r["section"]
    expected = "\u2605" if sec["id"] in task["expected_sections"] else " "
    kw_str = ", ".join(r["matched_keywords"]) if r["matched_keywords"] else "(none)"
    print(f"\u2502 {expected} {sec['title']:<35} score={r['combined']:.3f}  kw: {kw_str}")
print("\u2514" + "\u2500" * 58 + "\u2518")
print(f"  \u2605 = expected section")

# Step 2: Token savings
total_tokens = len(RETRIEVAL_SECTIONS) * ESTIMATED_TOKENS_PER_SECTION
selected_tokens = len(retrieved) * ESTIMATED_TOKENS_PER_SECTION
savings_pct = (1 - selected_tokens / total_tokens) * 100
print(f"\nToken Usage: {selected_tokens} / {total_tokens} (\u2193{savings_pct:.0f}% saved)")

# Step 3: Generate code with FULL context vs SELECTIVE context
full_context = "# Project CLAUDE.md\n\n" + "\n\n".join(s["content"] for s in RETRIEVAL_SECTIONS)
selective_context = "# Project CLAUDE.md (relevant sections)\n\n" + "\n\n".join(
    r["section"]["content"] for r in retrieved
)

print("\n\u23f3 Generating code with FULL context (all 12 sections)...")
full_prompt = f"You are a senior TypeScript developer. Follow ALL project conventions below.\n\n{full_context}"
resp_full = call_gemini(full_prompt, task["prompt"])
score_full = score_retrieval_output(resp_full, task["score_checks"])

print("\u23f3 Generating code with SELECTIVE context (top sections only)...")
sel_prompt = f"You are a senior TypeScript developer. Follow ALL project conventions below.\n\n{selective_context}"
resp_selective = call_gemini(sel_prompt, task["prompt"])
score_selective = score_retrieval_output(resp_selective, task["score_checks"])

# Step 4: Compare
print("\n" + "=" * 60)
print(f"LAB 5 RESULTS: {task['title']}")
print("=" * 60)
print(f"  {'Metric':<25} {'Full Context':>15} {'Selective':>15}")
print(f"  {'-'*55}")
print(f"  {'Sections used':<25} {len(RETRIEVAL_SECTIONS):>15} {len(retrieved):>15}")
print(f"  {'Est. tokens':<25} {total_tokens:>15} {selected_tokens:>15}")
print(f"  {'Token savings':<25} {'0%':>15} {f'{savings_pct:.0f}%':>15}")
print(f"  {'Checks matched':<25} {score_full['score']:>13}/{score_full['max_score']} {score_selective['score']:>13}/{score_selective['max_score']}")

print(f"\n  Full context matched:      {', '.join(score_full['matched']) or '(none)'}")
print(f"  Selective matched:         {', '.join(score_selective['matched']) or '(none)'}")

if score_selective["score"] >= score_full["score"]:
    print(f"\n  \u2728 Selective retrieval matched or beat full context with {savings_pct:.0f}% fewer tokens!")
else:
    diff = score_full["score"] - score_selective["score"]
    print(f"\n  \u26a0\ufe0f Selective missed {diff} check(s), but saved {savings_pct:.0f}% tokens.")
    print(f"     Missed: {', '.join(score_selective['missed'])}")

### Lab 5 Key Insight

Selective retrieval typically achieves **comparable or better** scores while using only **30–40%** of the token budget. This works because:

1. **Relevant context is concentrated** — the model focuses on what matters
2. **Irrelevant sections add noise** — more context isn’t always better
3. **Token savings compound** — at scale, this reduces cost and latency significantly

Try changing `TASK_INDEX` above (0–4) to see how retrieval works across different coding tasks.

> **Takeaway:** In production, always retrieve selectively. Even simple scoring (Jaccard + keywords) dramatically reduces waste.